# Phase 9-C Step 1: プロンプト改善評価

**作成日**: 2026-02-25  
**プロジェクト**: experiments-local-llm  
**目的**: プロンプト改善による回答品質向上の効果測定 (C0 vs C1)

---

## 実験設計

| ID | 構成 | プロンプト | モデル | 対象 |
|----|------|-----------|--------|------|
| **C0** | ベースライン | 現行版 | Qwen2.5-7B 4bit | Phase 9-B結果から取得 |
| **C1** | Step 1 | **改善版** | Qwen2.5-7B 4bit | 本ノートブックで評価 |

## 改善ポイント

- system_prompt: 3段構成（結論→根拠→補足）の回答構造を指示
- ユーザープロンプト: 固定回答開始文を構造ガイドに置換
- 推論過程・根拠引用・不確実性表現の誘導キーワードを埋め込み

## 目標値

| 指標 | C0 (現状) | C1目標 |
|------|----------|--------|
| composite_score | 52.2 | **60+** |
| reasoning_score | 1.43 | **2.0+** |
| evidence_score | 2.34 | **2.8+** |
| composite_success_rate | 33.85% | **45%+** |

## 1. 環境セットアップ

In [ ]:
# Google Colab環境チェック
import sys
import os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    PROJECT_PATH = '/content/drive/MyDrive/experiments-local-llm'
    sys.path.insert(0, f'{PROJECT_PATH}/src')
    
    # 必要なパッケージをインストール
    !pip install -q chromadb sentence-transformers networkx
    !pip install -q transformers accelerate bitsandbytes
    !pip install -q langchain langchain-core langchain-community langchain-chroma langgraph
    !pip install -q matplotlib seaborn pandas numpy tqdm
else:
    PROJECT_PATH = '..'
    sys.path.insert(0, f'{PROJECT_PATH}/src')

# 結果ディレクトリ
os.makedirs(f'{PROJECT_PATH}/results', exist_ok=True)

# GPU確認
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

print(f"\nProject path: {PROJECT_PATH}")
print(f"Running in Colab: {IN_COLAB}")

## 2. LLMロード（Qwen2.5-7B-Instruct 4bit）

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_community.llms import HuggingFacePipeline
import warnings
import gc
warnings.filterwarnings('ignore')

# 再実行時のVRAMクリーンアップ
for var_name in ['model', 'text_generation_pipeline', 'llm', 'embeddings']:
    if var_name in dir():
        try:
            del globals()[var_name]
        except KeyError:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM before load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

model_name = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {model_name}...")

# 4bit量子化設定
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("Tokenizer loaded")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("Model loaded (4-bit quantized)")

text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.0,
    do_sample=False,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

if torch.cuda.is_available():
    print(f"VRAM after LLM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("LLM pipeline ready")

## 3. 埋め込みモデル（multilingual-e5-base）

In [ ]:
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model_name = "intfloat/multilingual-e5-base"
print(f"Loading embedding model: {embedding_model_name}...")

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print("Embedding model loaded")

## 4. POIデータ読み込み（4エリア + combined）

In [ ]:
import json
from geo_utils import STATIONS, AREA_STATION_MAP, enrich_all_areas

# エリア設定
areas_config = {
    "shibuya": {
        "name": "渋谷駅周辺",
        "station": STATIONS["渋谷駅"]
    },
    "shinjuku": {
        "name": "新宿駅周辺",
        "station": STATIONS["新宿駅"]
    },
    "ikebukuro": {
        "name": "池袋駅周辺",
        "station": STATIONS["池袋駅"]
    },
    "tokyo": {
        "name": "東京駅周辺",
        "station": STATIONS["東京駅"]
    }
}

# 統合POIデータ読み込み
poi_all_file = f"{PROJECT_PATH}/data/poi_all_areas.json"
print(f"Loading POI data from {poi_all_file}...")

with open(poi_all_file, "r", encoding="utf-8") as f:
    raw_pois = json.load(f)

# メタデータをフラット化
flat_pois = []
for poi in raw_pois:
    if "metadata" in poi:
        flat_poi = poi["metadata"].copy()
        flat_pois.append(flat_poi)
    else:
        flat_pois.append(poi)

print(f"Total raw POIs: {len(flat_pois)}")

# エリア別POI数を表示
area_counts = {}
for poi in flat_pois:
    area_key = poi.get("area_key", "unknown")
    area_counts[area_key] = area_counts.get(area_key, 0) + 1

print("\nPOI counts by area:")
for area, count in sorted(area_counts.items()):
    area_name = areas_config.get(area, {}).get("name", area)
    print(f"  {area_name}: {count}")
print(f"  Total: {sum(area_counts.values())}")

## 5. ChromaDBベクトルストア構築（5コレクション）

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
import gc

def create_documents(pois):
    ""”POIリストからLangChain Documentを作成"""
    docs = []
    for poi in pois:
        content = f"{poi.get('name', '')} {poi.get('category', '')} {poi.get('description', '')}"
        docs.append(Document(page_content=content, metadata=poi))
    return docs

# エリア別 + 全統合のベクトルストア作成
vectorstores = {}
pois_by_area = {}

for area_key in areas_config:
    area_pois = [p for p in flat_pois if p.get("area_key") == area_key]
    pois_by_area[area_key] = area_pois
    docs = create_documents(area_pois)
    vectorstores[area_key] = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        collection_name=f"pois_{area_key}"
    )
    print(f"  {area_key}: {len(docs)} documents")

# 全統合コレクション
all_docs = create_documents(flat_pois)
vectorstores["all"] = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    collection_name="pois_all"
)
print(f"  all: {len(all_docs)} documents")

print("\nVectorstore construction complete")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 6. POI空間情報付与（enrich_all_areas）

In [ ]:
from geo_utils import enrich_all_areas

# 全POIに空間情報を付与
all_pois_enriched = enrich_all_areas(flat_pois, areas_config)
print(f"Enriched {len(all_pois_enriched)} POIs with spatial info")

# エリア別にも保持
enriched_by_area = {}
for poi in all_pois_enriched:
    area_key = poi.get("area_key", "unknown")
    enriched_by_area.setdefault(area_key, []).append(poi)

for area_key, area_pois in sorted(enriched_by_area.items()):
    if area_pois:
        avg_dist = sum(p.get("distance_from_station", 0) for p in area_pois) / len(area_pois)
        print(f"  {area_key}: {len(area_pois)} POIs, avg distance: {avg_dist:.0f}m")

## 7. テストケース読み込み

In [ ]:
from test_cases_multi_area import (
    ALL_MULTI_AREA_TEST_CASES,
    get_quick_test_cases,
    get_test_case_stats,
)

print(f"Total test cases: {len(ALL_MULTI_AREA_TEST_CASES)}")

# 統計表示
stats = get_test_case_stats()
print(f"\nBy area:")
for area, count in stats.get('by_area', {}).items():
    print(f"  {area}: {count}")

print(f"\nBy level:")
for level, count in sorted(stats.get('by_level', {}).items()):
    print(f"  L{level}: {count}")

print(f"\nQuick test cases: {len(get_quick_test_cases())}")

## 8. RAGシステム初期化（C1改善プロンプト適用済み）

**重要**: `structured_rag_system.py`, `adaptive_rag_system.py`, `agent_prompts.py` は
Phase 9-C Step 1 で改善済み。インポートするだけでC1プロンプトが適用される。

`graph_fn` はGraphRAGがコンテキストのみ返すため、LLM回答生成をノートブック内で行う。
これもC1改善プロンプトに更新。

In [ ]:
from structured_rag_system import StructuredRAGSystem
from agentic_rag_system import AgenticRAGSystem
from graph_rag_system import GraphRAGSystem
from adaptive_rag_system import AdaptiveRAGSystem
from agent_tools import set_global_pois_multi_area
from geo_utils import detect_target_area

# グローバルPOI設定（Agentic RAGのツール用）
set_global_pois_multi_area(all_pois_enriched, areas_config)
print("Global POIs set for multi-area tools")

# Hybrid RAG (StructuredRAG) - C1プロンプトが自動適用
print("\nInitializing Hybrid RAG (C1 prompt)...")
hybrid_system = StructuredRAGSystem(
    model=model,
    tokenizer=tokenizer,
    vectorstore=vectorstores["all"],
    all_pois=all_pois_enriched,
    areas_config=areas_config,
    debug=False
)
print("Hybrid RAG initialized")
# C1プロンプト確認
assert "結論" in hybrid_system.system_prompt, "C1プロンプトが適用されていません"
assert "根拠" in hybrid_system.system_prompt, "C1プロンプトが適用されていません"
print("✅ C1プロンプト確認 OK")

# Graph RAG
print("\nInitializing Graph RAG...")
graph_system = GraphRAGSystem(
    areas_config=areas_config,
    all_pois=all_pois_enriched
)
print(f"Graph RAG initialized ({len(graph_system.graphs)} area graphs)")

# Adaptive RAG - C1プロンプトが自動適用
print("\nInitializing Adaptive RAG (C1 prompt)...")
adaptive_system = AdaptiveRAGSystem(
    model=model,
    tokenizer=tokenizer,
    vectorstore=vectorstores["all"],
    all_pois=all_pois_enriched,
    areas_config=areas_config,
    verbose=False
)
print("Adaptive RAG initialized")

# Agentic RAG - C1プロンプトが自動適用
print("\nInitializing Agentic RAG (C1 prompt)...")
agentic_system = AgenticRAGSystem(
    model=model,
    tokenizer=tokenizer,
    model_name=model_name,
    verbose=False,
    max_iterations=5
)
print("Agentic RAG initialized")

print("\n✅ All systems initialized with C1 improved prompts")

In [ ]:
# システムラッパー関数（system_fn仕様）

def hybrid_fn(question: str) -> dict:
    """Hybrid RAG system_fn wrapper (C1 prompt)"""
    result = hybrid_system.query(question)
    detected = result.get("detected_area") or detect_target_area(question)
    return {
        "answer": result.get("answer", ""),
        "detected_area": detected,
    }


def graph_fn(question: str) -> dict:
    """Graph RAG system_fn wrapper - C1改善プロンプト適用"""
    graph_result = graph_system.query(question)
    context = graph_result.context

    # C1改善プロンプトでLLM回答生成
    area_names = "、".join(
        info.get("name", key) for key, info in areas_config.items()
    )
    system_prompt = f"""あなたは東京都内の主要駅周辺エリア（{area_names}）の地理情報に詳しいアシスタントです。
提供されたデータに基づいて、以下の構造で回答してください。

# 回答の構造
1. **結論**: 質問への直接的な回答を最初に述べる
2. **根拠**: データから得られた具体的な証拠を引用する
3. **補足**: 注意点や不確実な点があれば述べる

# 回答ルール
- 推論過程を明示する: 「したがって」「比較すると」「分析すると」「なぜなら」等の論理接続詞を使い、結論に至る過程を示す
- 根拠を具体的に引用する: POI名、座標(緯度, 経度)、距離(m)、件数を提供データから引用し、「データから」「検索結果に基づき」等で出典を明記する
- 数値は単位付きで示す: 距離はm、件数は件、座標は(35.xxx, 139.xxx)の形式で記載する
- 比較表現を使う: 「より多い」「最も近い」「〜倍」等の比較表現で差異を明確にする
- 不確実性を正直に示す: データで確認できない点は「ただし」「データの限界として」「可能性があります」「データからは確認できません」等で明記する
- 情報がない場合は「提供データからは確認できません」と正直に回答する"""

    prompt = f"""以下の提供データを参考にして、質問に回答してください。

{context}

【質問】
{question}

以下の構造で回答してください:
【結論】質問への直接的な回答
【根拠】データから引用した具体的なPOI名、距離(m)、件数等の証拠
【補足】データの限界や注意点（該当する場合）"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        parts = response.split("assistant")
        if len(parts) > 1:
            response = parts[-1].strip()

    del inputs, outputs
    torch.cuda.empty_cache()

    detected = detect_target_area(question)
    return {
        "answer": response,
        "detected_area": detected,
    }


def adaptive_fn(question: str) -> dict:
    """Adaptive RAG system_fn wrapper (C1 prompt)"""
    result = adaptive_system.query(question)
    detected = detect_target_area(question)
    return {
        "answer": result.response,
        "detected_area": detected,
    }


def agentic_fn(question: str) -> dict:
    """Agentic RAG system_fn wrapper (C1 prompt)"""
    result = agentic_system.query(question)
    detected = result.get("detected_area") or detect_target_area(question)
    return {
        "answer": result.get("answer", ""),
        "detected_area": detected,
    }


# 全システムの定義
SYSTEMS = {
    "hybrid_rag": hybrid_fn,
    "graph_rag": graph_fn,
    "adaptive_rag": adaptive_fn,
    "agentic_rag": agentic_fn,
}

print(f"\n{len(SYSTEMS)} systems ready for evaluation (C1 prompts)")

## 9. Quick Test実行（13件、Hybrid RAGのみ）

改善プロンプトの動作確認。回答に「結論」「根拠」「補足」構造が現れるかを確認。

In [ ]:
from evaluators_multi_area import MultiAreaEvaluator
from tqdm import tqdm
import gc

def clear_memory():
    """VRAM/RAM解放"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

evaluator = MultiAreaEvaluator(areas_config=areas_config, all_pois=all_pois_enriched)

# Quick Testケース
quick_cases = get_quick_test_cases()
print(f"Quick Test: {len(quick_cases)} cases (Hybrid RAG only)")

# Hybrid RAGのみQuick Test
quick_results = evaluator.evaluate_all(
    system_name="hybrid_rag",
    system_fn=hybrid_fn,
    test_cases=quick_cases
)

# スコア再計算
quick_results = evaluator.recalculate_scores(quick_results, quick_cases)

# サマリー表示
summary = evaluator.generate_summary(quick_results)
overall = summary["overall"]
print(f"\n{'='*60}")
print(f"Quick Test Results (Hybrid RAG, C1 prompt)")
print(f"{'='*60}")
print(f"  Success Rate:          {overall['success_rate']*100:.1f}%")
print(f"  Avg Keyword Hit Rate:  {overall['avg_keyword_hit_rate']:.3f}")
print(f"  Avg Composite Score:   {overall['avg_composite_score']:.1f}")
print(f"  Composite Success Rate:{overall['composite_success_rate']*100:.1f}%")
print(f"  Avg Reasoning Score:   {overall.get('avg_reasoning_score', 0):.2f}")
print(f"  Avg Evidence Score:    {overall.get('avg_evidence_score', 0):.2f}")
print(f"  Avg Time: {overall['avg_time_sec']:.1f}s")

clear_memory()

In [ ]:
# Quick Testの回答サンプル確認（C1プロンプトの効果を目視確認）
print("="*60)
print("回答サンプル確認（「結論」「根拠」「補足」構造が現れるか）")
print("="*60)

for r in quick_results[:5]:
    print(f"\n--- {r.test_id} (composite={r.composite_score}, reasoning={r.reasoning_score}, evidence={r.evidence_score}) ---")
    print(f"Q: {r.test_id}")
    answer_preview = r.answer[:500] if r.answer else "(no answer)"
    print(f"A: {answer_preview}")
    print()

## 10. Full Test実行: C1 Hybrid RAG × 130件（主評価）

**注意**: 130ケースの評価には約1-2時間かかります。
チェックポイント付きなので、中断後の再開が可能です。

In [ ]:
run_full_test = True  # Full Testを実行

full_results = {}

if run_full_test:
    all_cases = ALL_MULTI_AREA_TEST_CASES
    
    # === 主評価: Hybrid RAG × 130件 ===
    print(f"{'='*60}")
    print(f"C1 Evaluation: hybrid_rag (130 cases)")
    print(f"{'='*60}")
    
    checkpoint = f"{PROJECT_PATH}/results/checkpoint_c1_hybrid_rag.json"
    results = evaluator.evaluate_all(
        system_name="hybrid_rag",
        system_fn=hybrid_fn,
        test_cases=all_cases,
        checkpoint_file=checkpoint
    )
    full_results["hybrid_rag"] = results
    
    summary = evaluator.generate_summary(results)
    overall = summary["overall"]
    print(f"\n  Success Rate: {overall['success_rate']*100:.1f}%")
    print(f"  Avg Hit Rate: {overall['avg_keyword_hit_rate']:.3f}")
    print(f"  Avg Time: {overall['avg_time_sec']:.1f}s")
    
    clear_memory()
    print("\nHybrid RAG evaluation complete!")
else:
    print("Full Test skipped (set run_full_test = True to run)")

## 11. Full Test実行: C1 全システム × 130件（汎用性確認）

Step 1のみ、プロンプト改善の汎用性を確認するため**4システム×130件=520クエリ**の全量評価も実施。

**注意**: 520クエリの評価には数時間かかります。
先にHybrid RAGの結果を確認してから実行してください。

In [ ]:
run_all_systems = True  # 全システム評価を実行

if run_all_systems:
    all_cases = ALL_MULTI_AREA_TEST_CASES
    
    # hybrid_ragは既に実行済みなのでスキップ
    remaining_systems = {k: v for k, v in SYSTEMS.items() if k not in full_results}
    print(f"Remaining systems to evaluate: {list(remaining_systems.keys())}")
    
    for sys_name, sys_fn in remaining_systems.items():
        print(f"\n{'='*60}")
        print(f"C1 Evaluation: {sys_name} (130 cases)")
        print(f"{'='*60}")
        
        checkpoint = f"{PROJECT_PATH}/results/checkpoint_c1_{sys_name}.json"
        results = evaluator.evaluate_all(
            system_name=sys_name,
            system_fn=sys_fn,
            test_cases=all_cases,
            checkpoint_file=checkpoint
        )
        full_results[sys_name] = results
        
        summary = evaluator.generate_summary(results)
        overall = summary["overall"]
        print(f"\n  Success Rate: {overall['success_rate']*100:.1f}%")
        print(f"  Avg Hit Rate: {overall['avg_keyword_hit_rate']:.3f}")
        print(f"  Avg Time: {overall['avg_time_sec']:.1f}s")
        
        clear_memory()
    
    print(f"\nAll {len(SYSTEMS)} systems evaluated!")
else:
    print("All-systems evaluation skipped (set run_all_systems = True to run)")

## 12. 事後スコア計算（多次元評価）

In [ ]:
# 事後スコア計算（既存チェックポイント結果に多次元スコアを付与）
from test_cases_multi_area import ALL_MULTI_AREA_TEST_CASES

for sys_name, results in full_results.items():
    full_results[sys_name] = evaluator.recalculate_scores(results, ALL_MULTI_AREA_TEST_CASES)
    sample = full_results[sys_name][0]
    print(f"{sys_name}: composite={sample.composite_score}, reasoning={sample.reasoning_score}, "
          f"evidence={sample.evidence_score}, constraint={sample.constraint_score}")

print("\nMulti-dimensional scores recalculated for all systems")

## 13. C0ベースライン読み込み（Phase 9-B結果）

Phase 9-Bの評価結果をC0ベースラインとして読み込み、C1との差分を比較。

In [ ]:
import glob

# Phase 9-Bの最新結果ファイルを検索
baseline_files = sorted(glob.glob(f"{PROJECT_PATH}/results/phase9b_evaluation_*.json"))
if baseline_files:
    baseline_file = baseline_files[-1]  # 最新の結果を使用
    print(f"Loading C0 baseline from: {baseline_file}")
    
    with open(baseline_file, 'r', encoding='utf-8') as f:
        c0_data = json.load(f)
    
    # C0サマリー抽出
    c0_summaries = {}
    for sys_name, sys_data in c0_data.get("systems", {}).items():
        c0_summaries[sys_name] = sys_data["summary"]
    
    print(f"\nC0 baseline loaded: {len(c0_summaries)} systems")
    print(f"Test count: {c0_data.get('test_count', 'N/A')}")
    
    # C0サマリー表示
    print(f"\n{'='*80}")
    print(f"C0 Baseline (Phase 9-B)")
    print(f"{'='*80}")
    header = f"{'System':<20} {'Success%':<10} {'Composite':<10} {'CompSucc%':<10} {'Reasoning':<10} {'Evidence':<10}"
    print(header)
    print("-"*80)
    for sys_name, summary in c0_summaries.items():
        o = summary["overall"]
        print(f"{sys_name:<20} {o['success_rate']*100:<10.1f} {o['avg_composite_score']:<10.1f} "
              f"{o['composite_success_rate']*100:<10.1f} {o.get('avg_reasoning_score', 0):<10.2f} "
              f"{o.get('avg_evidence_score', 0):<10.2f}")
else:
    print("WARNING: No Phase 9-B baseline results found!")
    print("Expected files matching: results/phase9b_evaluation_*.json")
    c0_summaries = {}

## 14. C0 vs C1 比較分析（メイン結果）

In [ ]:
import pandas as pd
import numpy as np

# C1サマリー生成
c1_summaries = {}
for sys_name, results in full_results.items():
    c1_summaries[sys_name] = evaluator.generate_summary(results)

# === C0 vs C1 比較表 ===
print("="*100)
print("C0 (Phase 9-B Baseline) vs C1 (Prompt Improvement) Comparison")
print("="*100)

metrics = [
    ("success_rate", "Success Rate", 100, "%"),
    ("avg_composite_score", "Composite Score", 1, ""),
    ("composite_success_rate", "Composite Success%", 100, "%"),
    ("avg_reasoning_score", "Reasoning Score", 1, ""),
    ("avg_evidence_score", "Evidence Score", 1, ""),
]

for sys_name in full_results:
    if sys_name not in c0_summaries:
        continue
    
    c0 = c0_summaries[sys_name]["overall"]
    c1 = c1_summaries[sys_name]["overall"]
    
    print(f"\n--- {sys_name} ---")
    print(f"{'Metric':<25} {'C0':>10} {'C1':>10} {'Delta':>10} {'Status':>8}")
    print("-"*65)
    
    for key, label, multiplier, unit in metrics:
        v0 = c0.get(key, 0) * multiplier
        v1 = c1.get(key, 0) * multiplier
        delta = v1 - v0
        status = "↑" if delta > 0 else ("↓" if delta < 0 else "→")
        print(f"{label:<25} {v0:>9.1f}{unit} {v1:>9.1f}{unit} {delta:>+9.1f}{unit} {status:>8}")

# === Hybrid RAGのC1目標達成判定 ===
if "hybrid_rag" in c1_summaries:
    c1_hybrid = c1_summaries["hybrid_rag"]["overall"]
    print(f"\n{'='*60}")
    print(f"C1 Target Achievement (Hybrid RAG)")
    print(f"{'='*60}")
    
    targets = [
        ("avg_composite_score", "Composite Score", 60.0, 1),
        ("avg_reasoning_score", "Reasoning Score", 2.0, 1),
        ("avg_evidence_score", "Evidence Score", 2.8, 1),
        ("composite_success_rate", "Composite Success%", 0.45, 100),
    ]
    
    all_met = True
    for key, label, target, multiplier in targets:
        value = c1_hybrid.get(key, 0)
        display_val = value * multiplier
        display_target = target * multiplier
        met = value >= target
        if not met:
            all_met = False
        status = "✅ PASS" if met else "❌ MISS"
        print(f"  {label:<25} {display_val:>8.1f} / {display_target:.1f}  {status}")
    
    print(f"\n  Overall: {'ALL TARGETS MET' if all_met else 'SOME TARGETS MISSED'}")

## 15. レベル別・エリア別 C0 vs C1 比較

In [ ]:
# === Hybrid RAGのレベル別比較 ===
if "hybrid_rag" in c0_summaries and "hybrid_rag" in c1_summaries:
    print("="*80)
    print("Hybrid RAG: Level-wise C0 vs C1 Comparison")
    print("="*80)
    
    c0_by_level = c0_summaries["hybrid_rag"].get("by_level", {})
    c1_by_level = c1_summaries["hybrid_rag"].get("by_level", {})
    
    level_names = {1: "L1 Basic", 2: "L2 Spatial", 3: "L3 Constraint", 4: "L4 Decision", 5: "L5 Advanced"}
    
    print(f"{'Level':<15} {'C0 Composite':>12} {'C1 Composite':>12} {'Delta':>8} {'C0 Success%':>12} {'C1 Success%':>12} {'Delta':>8}")
    print("-"*80)
    
    for level in [1, 2, 3, 4, 5]:
        c0_l = c0_by_level.get(level, c0_by_level.get(str(level), {}))
        c1_l = c1_by_level.get(level, c1_by_level.get(str(level), {}))
        
        c0_comp = c0_l.get("avg_composite_score", 0)
        c1_comp = c1_l.get("avg_composite_score", 0)
        c0_succ = c0_l.get("success_rate", 0) * 100
        c1_succ = c1_l.get("success_rate", 0) * 100
        
        print(f"{level_names.get(level, f'L{level}'):<15} {c0_comp:>12.1f} {c1_comp:>12.1f} {c1_comp-c0_comp:>+8.1f} "
              f"{c0_succ:>11.1f}% {c1_succ:>11.1f}% {c1_succ-c0_succ:>+7.1f}%")

# === Hybrid RAGのエリア別比較 ===
if "hybrid_rag" in c0_summaries and "hybrid_rag" in c1_summaries:
    print(f"\n{'='*80}")
    print("Hybrid RAG: Area-wise C0 vs C1 Comparison")
    print("="*80)
    
    c0_by_area = c0_summaries["hybrid_rag"].get("by_area", {})
    c1_by_area = c1_summaries["hybrid_rag"].get("by_area", {})
    
    print(f"{'Area':<15} {'C0 Composite':>12} {'C1 Composite':>12} {'Delta':>8} {'C0 Success%':>12} {'C1 Success%':>12} {'Delta':>8}")
    print("-"*80)
    
    for area in sorted(set(list(c0_by_area.keys()) + list(c1_by_area.keys()))):
        c0_a = c0_by_area.get(area, {})
        c1_a = c1_by_area.get(area, {})
        
        c0_comp = c0_a.get("avg_composite_score", 0)
        c1_comp = c1_a.get("avg_composite_score", 0)
        c0_succ = c0_a.get("success_rate", 0) * 100
        c1_succ = c1_a.get("success_rate", 0) * 100
        
        print(f"{area:<15} {c0_comp:>12.1f} {c1_comp:>12.1f} {c1_comp-c0_comp:>+8.1f} "
              f"{c0_succ:>11.1f}% {c1_succ:>11.1f}% {c1_succ-c0_succ:>+7.1f}%")

## 16. 全システム C1 結果比較

In [ ]:
# 全システムC1比較
comparison = evaluator.compare_systems(full_results)

print("="*100)
print("C1 Overall System Comparison (All Systems with Improved Prompts)")
print("="*100)

header = (f"{'System':<20} {'Success%':<10} {'AvgHitRate':<11} {'Composite':<10} "
          f"{'CompSucc%':<10} {'Reasoning':<10} {'Evidence':<10} {'AvgTime(s)':<11}")
print(header)
print("-"*100)

for sys_name in full_results:
    s = comparison["summaries"][sys_name]["overall"]
    print(f"{sys_name:<20} {s['success_rate']*100:<10.1f} {s['avg_keyword_hit_rate']:<11.3f} "
          f"{s['avg_composite_score']:<10.1f} {s['composite_success_rate']*100:<10.1f} "
          f"{s.get('avg_reasoning_score', 0):<10.2f} {s.get('avg_evidence_score', 0):<10.2f} "
          f"{s['avg_time_sec']:<11.1f}")

# ランキング
print(f"\nRankings (by composite score):")
ranked = sorted(
    full_results.keys(),
    key=lambda s: comparison["summaries"][s]["overall"]["avg_composite_score"],
    reverse=True
)
for i, name in enumerate(ranked, 1):
    comp = comparison["summaries"][name]["overall"]["avg_composite_score"]
    succ = comparison["summaries"][name]["overall"]["success_rate"] * 100
    print(f"  {i}. {name}: composite={comp:.1f}, success={succ:.1f}%")

## 17. 可視化

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

# --- Fig 1: C0 vs C1 Composite Score Comparison (Hybrid RAG) ---
if "hybrid_rag" in c0_summaries and "hybrid_rag" in c1_summaries:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Composite Score by Level
    c0_by_level = c0_summaries["hybrid_rag"].get("by_level", {})
    c1_by_level = c1_summaries["hybrid_rag"].get("by_level", {})
    
    levels = [1, 2, 3, 4, 5]
    level_labels = ['L1\nBasic', 'L2\nSpatial', 'L3\nConstraint', 'L4\nDecision', 'L5\nAdvanced']
    x = np.arange(len(levels))
    width = 0.35
    
    c0_scores = [c0_by_level.get(l, c0_by_level.get(str(l), {})).get('avg_composite_score', 0) for l in levels]
    c1_scores = [c1_by_level.get(l, c1_by_level.get(str(l), {})).get('avg_composite_score', 0) for l in levels]
    
    bars1 = axes[0].bar(x - width/2, c0_scores, width, label='C0 (Baseline)', color='#95a5a6', alpha=0.8)
    bars2 = axes[0].bar(x + width/2, c1_scores, width, label='C1 (Improved)', color='#3498db', alpha=0.8)
    
    axes[0].set_xlabel('Level')
    axes[0].set_ylabel('Composite Score')
    axes[0].set_title('Hybrid RAG: Composite Score by Level (C0 vs C1)')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(level_labels)
    axes[0].legend()
    axes[0].set_ylim([0, 100])
    axes[0].axhline(y=60, color='red', linestyle='--', alpha=0.5, label='Target (60)')
    
    # Delta bars
    deltas = [c1 - c0 for c0, c1 in zip(c0_scores, c1_scores)]
    colors = ['#2ecc71' if d > 0 else '#e74c3c' for d in deltas]
    axes[1].bar(x, deltas, color=colors, alpha=0.8)
    axes[1].set_xlabel('Level')
    axes[1].set_ylabel('Delta (C1 - C0)')
    axes[1].set_title('Composite Score Improvement by Level')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(level_labels)
    axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    for i, d in enumerate(deltas):
        axes[1].text(i, d + (1 if d >= 0 else -2), f'{d:+.1f}', ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'{PROJECT_PATH}/results/phase9c_step1_level_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

# --- Fig 2: C0 vs C1 Multi-dimensional Scores (Hybrid RAG) ---
if "hybrid_rag" in c0_summaries and "hybrid_rag" in c1_summaries:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    c0_o = c0_summaries["hybrid_rag"]["overall"]
    c1_o = c1_summaries["hybrid_rag"]["overall"]
    
    metrics_list = [
        ('Composite\nScore', c0_o.get('avg_composite_score', 0), c1_o.get('avg_composite_score', 0)),
        ('Reasoning\n(x20)', c0_o.get('avg_reasoning_score', 0)*20, c1_o.get('avg_reasoning_score', 0)*20),
        ('Evidence\n(x20)', c0_o.get('avg_evidence_score', 0)*20, c1_o.get('avg_evidence_score', 0)*20),
        ('Success\nRate', c0_o.get('composite_success_rate', 0)*100, c1_o.get('composite_success_rate', 0)*100),
    ]
    
    labels = [m[0] for m in metrics_list]
    c0_vals = [m[1] for m in metrics_list]
    c1_vals = [m[2] for m in metrics_list]
    
    x = np.arange(len(labels))
    width = 0.35
    
    ax.bar(x - width/2, c0_vals, width, label='C0 (Baseline)', color='#95a5a6', alpha=0.8)
    ax.bar(x + width/2, c1_vals, width, label='C1 (Improved)', color='#3498db', alpha=0.8)
    
    ax.set_ylabel('Score (0-100 scale)')
    ax.set_title('Hybrid RAG: Multi-dimensional Score Comparison (C0 vs C1)')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    ax.set_ylim([0, 100])
    
    # Add delta labels
    for i in range(len(labels)):
        delta = c1_vals[i] - c0_vals[i]
        max_val = max(c0_vals[i], c1_vals[i])
        ax.text(i, max_val + 2, f'{delta:+.1f}', ha='center', fontweight='bold',
                color='green' if delta > 0 else 'red')
    
    plt.tight_layout()
    plt.savefig(f'{PROJECT_PATH}/results/phase9c_step1_metrics_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

# --- Fig 3: All Systems C1 Composite Score ---
if len(full_results) > 1:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    sys_names = list(full_results.keys())
    composite_scores = [comparison["summaries"][s]["overall"]["avg_composite_score"] for s in sys_names]
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12'][:len(sys_names)]
    
    bars = ax.barh(sys_names, composite_scores, color=colors, alpha=0.8)
    ax.set_xlabel('Composite Score')
    ax.set_title('C1: Composite Score by System')
    ax.axvline(x=60, color='red', linestyle='--', alpha=0.5, label='Target (60)')
    ax.legend()
    
    for i, v in enumerate(composite_scores):
        ax.text(v + 1, i, f'{v:.1f}', va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'{PROJECT_PATH}/results/phase9c_step1_system_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

print("Visualizations saved")

## 18. 結果保存（JSON + テキストサマリー）

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# numpy型をPython標準型に変換
def convert_to_serializable(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(i) for i in obj]
    return obj

# JSON結果保存
results_data = {
    "experiment": "phase9c_step1",
    "description": "Prompt improvement evaluation (C0 vs C1)",
    "timestamp": timestamp,
    "test_count": len(next(iter(full_results.values()))),
    "model": "Qwen2.5-7B-Instruct 4bit",
    "prompt_version": "C1 (improved: 3-part structure with reasoning/evidence/uncertainty)",
    "systems": {}
}

for sys_name, results in full_results.items():
    summary = evaluator.generate_summary(results)
    results_data["systems"][sys_name] = {
        "summary": convert_to_serializable(summary),
        "results": [convert_to_serializable(r.to_dict()) for r in results]
    }

# C0ベースライン情報も含める
if c0_summaries:
    results_data["baseline_c0"] = convert_to_serializable({
        "source": baseline_file if 'baseline_file' in dir() else "N/A",
        "summaries": c0_summaries
    })

results_data["comparison"] = convert_to_serializable(comparison)

output_file = f"{PROJECT_PATH}/results/phase9c_step1_{timestamp}.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results_data, f, ensure_ascii=False, indent=2)
print(f"Results saved to {output_file}")

# テキストサマリー保存
summary_file = f"{PROJECT_PATH}/results/phase9c_step1_summary_{timestamp}.txt"
with open(summary_file, 'w', encoding='utf-8') as f:
    f.write("Phase 9-C Step 1: Prompt Improvement Evaluation Summary\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Test Cases: {results_data['test_count']}\n")
    f.write(f"Model: {results_data['model']}\n")
    f.write(f"Prompt: {results_data['prompt_version']}\n\n")
    
    f.write("C0 vs C1 Comparison (Hybrid RAG):\n")
    f.write("-" * 60 + "\n")
    if "hybrid_rag" in c0_summaries and "hybrid_rag" in c1_summaries:
        c0_o = c0_summaries["hybrid_rag"]["overall"]
        c1_o = c1_summaries["hybrid_rag"]["overall"]
        for key, label in [("avg_composite_score", "Composite"), ("avg_reasoning_score", "Reasoning"),
                           ("avg_evidence_score", "Evidence"), ("composite_success_rate", "CompSuccess%")]:
            v0 = c0_o.get(key, 0)
            v1 = c1_o.get(key, 0)
            mult = 100 if "rate" in key else 1
            f.write(f"  {label}: C0={v0*mult:.1f} -> C1={v1*mult:.1f} (delta={((v1-v0)*mult):+.1f})\n")
    
    f.write("\nAll Systems C1 Results:\n")
    f.write("-" * 60 + "\n")
    for sys_name in full_results:
        s = comparison["summaries"][sys_name]["overall"]
        f.write(f"  {sys_name}: composite={s['avg_composite_score']:.1f} "
                f"success={s['success_rate']*100:.1f}% "
                f"reasoning={s.get('avg_reasoning_score', 0):.2f} "
                f"evidence={s.get('avg_evidence_score', 0):.2f}\n")

print(f"Summary saved to {summary_file}")
print("\nEvaluation complete!")

## 19. 結論・次ステップ

### Step 1 評価結果の要約

| 指標 | C0 (現状) | C1目標 | C1実績 | 判定 |
|------|----------|--------|--------|------|
| composite_score | 52.2 | 60+ | **TBD** | TBD |
| reasoning_score | 1.43 | 2.0+ | **TBD** | TBD |
| evidence_score | 2.34 | 2.8+ | **TBD** | TBD |
| composite_success_rate | 33.85% | 45%+ | **TBD** | TBD |

### 次ステップ

- **Step 1成功時**: Step 2（モデル変更: Qwen2.5-14B 4bit等）へ進む
- **Step 1不十分時**: Few-shot例の追加、プロンプトパターンの変更を検討